# 01 — Krylov / Lanczos inexact CR

Compare exact CR vs Krylov subspace dimensions $m \in \{5,10,20,50\}$. Plot time per iteration vs dimension $n$.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from cubic_reg.problems import Quadratic
from cubic_reg.solvers import cr, krylov

%matplotlib inline

In [ ]:
p = Quadratic(n=80, condition=100.0, seed=0)
x0 = np.ones(p.dim)
fig, ax = plt.subplots(figsize=(7, 4))
r0 = cr.minimize(p, x0=x0, M=1.0, eps=1e-8)
ax.semilogy(np.maximum(np.asarray(r0.history_f) - p.f_star, 1e-16), label=f"exact CR nit={r0.nit}")
for m in [5, 10, 20, 50]:
    r = krylov.minimize(p, x0=x0, M=1.0, eps=1e-8, krylov_dim=m, max_iter=100)
    gap = np.maximum(np.asarray(r.history_f) - p.f_star, 1e-16)
    ax.semilogy(gap, label=f"m={m}, nit={r.nit}, hvp={r.n_hvp}")
ax.legend(); ax.set_xlabel("iteration"); ax.set_ylabel(r"$f-f^*$"); ax.grid(True, alpha=0.3)
plt.title("Exact vs Krylov CR"); plt.show()

In [ ]:
ns = [50, 100, 200, 400, 800]
times_exact, times_kry = [], []
for n in ns:
    p = Quadratic(n=n, condition=50.0, seed=0)
    x0 = np.ones(n)
    r_e = cr.minimize(p, x0=x0, M=1.0, eps=1e-6, max_iter=20)
    r_k = krylov.minimize(p, x0=x0, M=1.0, eps=1e-6, krylov_dim=20, max_iter=20)
    times_exact.append(r_e.time_sec / max(r_e.nit, 1))
    times_kry.append(r_k.time_sec / max(r_k.nit, 1))
plt.figure(figsize=(7, 4))
plt.loglog(ns, times_exact, "o-", label="exact CR / iter")
plt.loglog(ns, times_kry, "s-", label="Krylov m=20 / iter")
plt.xlabel("n"); plt.ylabel("sec / iteration"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.title("Scalability: time per iteration vs n"); plt.show()